# 06 — LLM RAG Assistant and Guardrails

## Responsible AI RAG Assistant

This notebook builds the final answer-generation layer of the Responsible AI RAG Assistant project.

The goal is to connect the evaluated multi-source retrieval system to an LLM-assisted answer generation workflow.

This notebook uses:

- a multi-source Chroma vector store
- source-aware retrieval
- retrieved context formatting
- responsible-use guardrails
- optional OpenAI answer generation
- fallback context-only answer generation when no API key is available

## Project Context

The assistant is designed as a portfolio-ready AI Engineering prototype for Responsible AI, EU AI Act, GDPR, and AI risk management knowledge support.

It uses public and authoritative sources only.

## Important Responsible-Use Note

This assistant does **not** provide legal advice.

Answers must be grounded in retrieved public-source context. If retrieved context is insufficient, the assistant should say that the available sources do not support a confident answer.

Real compliance decisions should be reviewed by qualified legal, privacy, or compliance professionals.

In [1]:
from pathlib import Path
from datetime import date
import os
import re
import textwrap

import pandas as pd
import numpy as np

print("Notebook 06 setup started.")
print(f"Current date: {date.today()}")

Notebook 06 setup started.
Current date: 2026-07-04


## Project Paths

This section defines the project folder paths used to load processed chunks, embedding metadata, and the Chroma vector store created in the previous notebooks.

In [2]:
# Define project paths

current_path = Path.cwd()

if current_path.name == "notebooks":
    PROJECT_ROOT = current_path.parent
else:
    PROJECT_ROOT = current_path

DATA_DIR = PROJECT_ROOT / "data"
RAW_DATA_DIR = DATA_DIR / "raw"
PROCESSED_DATA_DIR = DATA_DIR / "processed"
VECTOR_STORE_DIR = DATA_DIR / "vector_store"
REPORTS_DIR = PROJECT_ROOT / "reports"
DOCS_DIR = PROJECT_ROOT / "docs"

paths = {
    "PROJECT_ROOT": PROJECT_ROOT,
    "DATA_DIR": DATA_DIR,
    "RAW_DATA_DIR": RAW_DATA_DIR,
    "PROCESSED_DATA_DIR": PROCESSED_DATA_DIR,
    "VECTOR_STORE_DIR": VECTOR_STORE_DIR,
    "REPORTS_DIR": REPORTS_DIR,
    "DOCS_DIR": DOCS_DIR,
}

for name, path in paths.items():
    print(f"{name}: {path}")
    print(f"Exists: {path.exists()}")
    print("-" * 80)

PROJECT_ROOT: c:\Users\mdadg\Desktop\responsible-ai-rag-assistant
Exists: True
--------------------------------------------------------------------------------
DATA_DIR: c:\Users\mdadg\Desktop\responsible-ai-rag-assistant\data
Exists: True
--------------------------------------------------------------------------------
RAW_DATA_DIR: c:\Users\mdadg\Desktop\responsible-ai-rag-assistant\data\raw
Exists: True
--------------------------------------------------------------------------------
PROCESSED_DATA_DIR: c:\Users\mdadg\Desktop\responsible-ai-rag-assistant\data\processed
Exists: True
--------------------------------------------------------------------------------
VECTOR_STORE_DIR: c:\Users\mdadg\Desktop\responsible-ai-rag-assistant\data\vector_store
Exists: True
--------------------------------------------------------------------------------
REPORTS_DIR: c:\Users\mdadg\Desktop\responsible-ai-rag-assistant\reports
Exists: True
-------------------------------------------------------------

## Load Retrieval Components

This section loads the same embedding model and multi-source Chroma collection created in Notebook 05.

The collection should contain 856 chunks from five authoritative public sources:

1. European Commission AI Act overview
2. Official EU AI Act legal text
3. Official GDPR legal text
4. NIST AI Risk Management Framework PDF
5. NIST AI RMF overview page

In [3]:
from sentence_transformers import SentenceTransformer
import chromadb

embedding_model_name = "sentence-transformers/all-MiniLM-L6-v2"
multisource_collection_name = "responsible_ai_rag_multisource_v1"

print(f"Loading embedding model: {embedding_model_name}")
embedding_model = SentenceTransformer(embedding_model_name)
print("Embedding model loaded.")

chroma_client = chromadb.PersistentClient(path=str(VECTOR_STORE_DIR))
multisource_collection = chroma_client.get_collection(
    name=multisource_collection_name
)

print(f"Loaded Chroma collection: {multisource_collection_name}")
print(f"Collection count: {multisource_collection.count()}")

Loading embedding model: sentence-transformers/all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded.
Loaded Chroma collection: responsible_ai_rag_multisource_v1
Collection count: 856


In [4]:
# Load supporting processed files

multisource_chunks_path = PROCESSED_DATA_DIR / "multisource_chunks.csv"
source_inventory_path = PROCESSED_DATA_DIR / "source_inventory.csv"
retrieval_comparison_path = PROCESSED_DATA_DIR / "retrieval_baseline_vs_source_aware_comparison.csv"

multisource_chunks_df = pd.read_csv(multisource_chunks_path)
source_inventory_df = pd.read_csv(source_inventory_path)
retrieval_comparison_df = pd.read_csv(retrieval_comparison_path)

print(f"Loaded chunks: {len(multisource_chunks_df)}")
print(f"Loaded sources: {source_inventory_df['source_id'].nunique()}")
print(f"Loaded retrieval comparison rows: {len(retrieval_comparison_df)}")

print("-" * 80)

display(
    source_inventory_df[
        [
            "source_id",
            "source_name",
            "publisher",
            "source_type",
            "collection_status",
            "processing_status",
        ]
    ]
)

Loaded chunks: 856
Loaded sources: 5
Loaded retrieval comparison rows: 8
--------------------------------------------------------------------------------


,source_id,source_name,publisher,source_type,collection_status,processing_status
0,SRC-001,AI Act,European Commission,Official web page,collected,text_extracted
1,SRC-002,Regulation (EU) 2024/1689 Artificial Intellige...,EUR-Lex / European Union,Official legal text,collected,text_extracted
2,SRC-003,Regulation (EU) 2016/679 General Data Protecti...,EUR-Lex / European Union,Official legal text,collected,text_extracted
3,SRC-004,Artificial Intelligence Risk Management Framew...,NIST,Public framework PDF,collected,text_extracted
4,SRC-005,AI Risk Management Framework overview page,NIST,Official web page,collected,text_extracted


## Source-Aware Retrieval

Notebook 05 showed that baseline semantic retrieval worked well, but source-aware reranking improved retrieval quality from:

- baseline strong results: 5 / 8
- source-aware strong results: 8 / 8

This section recreates the source-aware retrieval logic for the final RAG assistant.

The assistant uses lightweight query routing to prioritize the most relevant source family:

- EU AI Act questions → SRC-001 and SRC-002
- GDPR questions → SRC-003
- NIST / AI RMF questions → SRC-004 and SRC-005

This improves retrieval precision when legal and governance sources contain overlapping concepts.

In [5]:
def infer_preferred_sources(query: str) -> list[str]:
    """
    Infer likely preferred source families from the user query.

    This function does not generate an answer.
    It only helps the retrieval layer prioritize the most relevant source family
    when multiple governance/legal sources contain overlapping terminology.
    """
    query_lower = query.lower()

    gdpr_keywords = [
        "gdpr",
        "personal data",
        "data subject",
        "data protection",
        "lawful basis",
        "processing personal data",
        "controller",
        "processor",
        "consent",
        "right to erasure",
        "right of access",
    ]

    nist_keywords = [
        "nist",
        "ai rmf",
        "risk management framework",
        "govern",
        "map",
        "measure",
        "manage",
        "trustworthy ai",
        "ai risk management",
    ]

    ai_act_keywords = [
        "ai act",
        "high-risk",
        "high risk",
        "prohibited",
        "transparency",
        "ai-generated",
        "general-purpose ai",
        "general purpose ai",
        "risk-based approach",
        "conformity assessment",
        "provider obligations",
        "deployer obligations",
    ]

    if any(keyword in query_lower for keyword in gdpr_keywords):
        return ["SRC-003"]

    if any(keyword in query_lower for keyword in nist_keywords):
        return ["SRC-004", "SRC-005"]

    if any(keyword in query_lower for keyword in ai_act_keywords):
        return ["SRC-001", "SRC-002"]

    return []


# Quick checks
test_queries = [
    "What is personal data under the GDPR?",
    "What does NIST say about AI risk management?",
    "What is the risk-based approach of the EU AI Act?",
    "How should organizations govern AI systems?",
]

for query in test_queries:
    print(f"Query: {query}")
    print(f"Preferred sources: {infer_preferred_sources(query)}")
    print("-" * 80)

Query: What is personal data under the GDPR?
Preferred sources: ['SRC-003']
--------------------------------------------------------------------------------
Query: What does NIST say about AI risk management?
Preferred sources: ['SRC-004', 'SRC-005']
--------------------------------------------------------------------------------
Query: What is the risk-based approach of the EU AI Act?
Preferred sources: ['SRC-001', 'SRC-002']
--------------------------------------------------------------------------------
Query: How should organizations govern AI systems?
Preferred sources: ['SRC-004', 'SRC-005']
--------------------------------------------------------------------------------


In [6]:
def search_multisource_vector_store(query: str, n_results: int = 5) -> pd.DataFrame:
    """
    Search the multi-source Chroma vector store using a natural-language query.
    """
    query_embedding = embedding_model.encode(
        query,
        convert_to_numpy=True,
        normalize_embeddings=True,
    )

    results = multisource_collection.query(
        query_embeddings=[query_embedding.tolist()],
        n_results=n_results,
        include=["documents", "metadatas", "distances"],
    )

    retrieved_records = []

    for rank, chunk_id in enumerate(results["ids"][0], start=1):
        metadata = results["metadatas"][0][rank - 1]
        document = results["documents"][0][rank - 1]
        distance = results["distances"][0][rank - 1]

        retrieved_records.append(
            {
                "rank": rank,
                "chunk_id": chunk_id,
                "distance": float(distance),
                "source_id": metadata.get("source_id", ""),
                "source_name": metadata.get("source_name", ""),
                "publisher": metadata.get("publisher", ""),
                "source_type": metadata.get("source_type", ""),
                "main_topic": metadata.get("main_topic", ""),
                "chunk_index": metadata.get("chunk_index", ""),
                "word_count": metadata.get("word_count", ""),
                "url": metadata.get("url", ""),
                "retrieved_text": document,
            }
        )

    return pd.DataFrame(retrieved_records)


print("Baseline multi-source retrieval function is ready.")

Baseline multi-source retrieval function is ready.


In [7]:
def search_multisource_vector_store_source_aware(
    query: str,
    n_results: int = 5,
    n_initial_results: int = 20,
) -> pd.DataFrame:
    """
    Search the multi-source vector store and apply source-aware reranking.

    Workflow:
    1. Retrieve a broader candidate set from Chroma.
    2. Infer preferred source family from the query.
    3. If preferred sources are detected, prioritize those chunks.
    4. Preserve semantic distance ordering within the preferred source group.
    """
    preferred_sources = infer_preferred_sources(query)

    initial_results_df = search_multisource_vector_store(
        query=query,
        n_results=n_initial_results,
    )

    initial_results_df["preferred_source"] = initial_results_df["source_id"].isin(
        preferred_sources
    )

    if preferred_sources and initial_results_df["preferred_source"].any():
        reranked_df = initial_results_df.sort_values(
            by=["preferred_source", "distance"],
            ascending=[False, True],
        ).head(n_results)
    else:
        reranked_df = initial_results_df.sort_values(
            by="distance",
            ascending=True,
        ).head(n_results)

    reranked_df = reranked_df.copy()
    reranked_df["rank"] = range(1, len(reranked_df) + 1)
    reranked_df["preferred_sources"] = ", ".join(preferred_sources)

    return reranked_df.reset_index(drop=True)


print("Source-aware retrieval function is ready.")

Source-aware retrieval function is ready.


In [8]:
# Test source-aware retrieval with representative questions

test_questions = [
    "What is the risk-based approach of the EU AI Act?",
    "What is personal data under the GDPR?",
    "What does NIST say about managing AI risks?",
]

for question in test_questions:
    print("=" * 100)
    print(f"Question: {question}")
    print(f"Preferred sources: {infer_preferred_sources(question)}")
    print("-" * 100)

    results_df = search_multisource_vector_store_source_aware(
        query=question,
        n_results=5,
        n_initial_results=20,
    )

    display(
        results_df[
            [
                "rank",
                "distance",
                "source_id",
                "source_name",
                "publisher",
                "chunk_index",
                "word_count",
                "preferred_source",
                "retrieved_text",
            ]
        ]
    )

Question: What is the risk-based approach of the EU AI Act?
Preferred sources: ['SRC-001', 'SRC-002']
----------------------------------------------------------------------------------------------------


,rank,distance,source_id,source_name,publisher,chunk_index,word_count,preferred_source,retrieved_text
0,1,0.525325,SRC-001,AI Act,European Commission,2,300,True,The AI Act ensures that Europeans can trust wh...
1,2,0.537072,SRC-001,AI Act,European Commission,1,300,True,The AI Act is the first-ever legal framework o...
2,3,0.735104,SRC-001,AI Act,European Commission,8,300,True,set for the rules governing high-risk AI syste...
3,4,0.800995,SRC-001,AI Act,European Commission,7,300,True,the second quarter of 2026. Find out more abou...
4,5,0.810517,SRC-001,AI Act,European Commission,3,300,True,the practical application of the prohibited pr...


Question: What is personal data under the GDPR?
Preferred sources: ['SRC-003']
----------------------------------------------------------------------------------------------------


,rank,distance,source_id,source_name,publisher,chunk_index,word_count,preferred_source,retrieved_text
0,1,0.807489,SRC-003,Regulation (EU) 2016/679 General Data Protecti...,EUR-Lex / European Union,56,300,True,ies of personal data should be allo wed only u...
1,2,0.813962,SRC-003,Regulation (EU) 2016/679 General Data Protecti...,EUR-Lex / European Union,4,300,True,of personal data. The exc hange of personal da...
2,3,0.815230,SRC-003,Regulation (EU) 2016/679 General Data Protecti...,EUR-Lex / European Union,133,300,True,data consisting of the use of personal data to...
3,4,0.841520,SRC-003,Regulation (EU) 2016/679 General Data Protecti...,EUR-Lex / European Union,132,300,True,U nion. 3. This Regulation applies to the proc...
4,5,0.885930,SRC-003,Regulation (EU) 2016/679 General Data Protecti...,EUR-Lex / European Union,3,300,True,the Offi cial Jour nal). P osition of the Euro...


Question: What does NIST say about managing AI risks?
Preferred sources: ['SRC-004', 'SRC-005']
----------------------------------------------------------------------------------------------------


,rank,distance,source_id,source_name,publisher,chunk_index,word_count,preferred_source,retrieved_text
0,1,0.403003,SRC-005,AI Risk Management Framework overview page,NIST,3,82,True,"FinRegLab, and the Stanford Institute for Huma..."
1,2,0.544553,SRC-005,AI Risk Management Framework overview page,NIST,2,300,True,aligns with their goals and priorities. On Apr...
2,3,0.610519,SRC-004,Artificial Intelligence Risk Management Framew...,NIST,9,300,True,and the Plan for Federal Engagement in Develop...
3,4,0.634200,SRC-004,Artificial Intelligence Risk Management Framew...,NIST,67,167,True,or aligned with other approaches to managing A...
4,5,0.652685,SRC-004,Artificial Intelligence Risk Management Framew...,NIST,33,300,True,in Artificial Intelligence.) Page 18 [Page 24]...


## Format Retrieved Context

The RAG assistant should not send the raw retrieval table directly to the answer-generation step.

Instead, retrieved chunks are converted into a structured context block that includes:

- source ID
- source name
- publisher
- chunk index
- URL
- retrieved text

This makes the answer-generation step more transparent, source-aware, and easier to audit.

In [9]:
def clean_text_for_prompt(text: str) -> str:
    """
    Clean retrieved text before inserting it into a RAG prompt.
    """
    if pd.isna(text):
        return ""

    cleaned = str(text)
    cleaned = re.sub(r"\s+", " ", cleaned).strip()
    return cleaned


def truncate_text(text: str, max_characters: int = 1200) -> str:
    """
    Truncate long retrieved text to keep the prompt readable.
    """
    cleaned = clean_text_for_prompt(text)

    if len(cleaned) <= max_characters:
        return cleaned

    return cleaned[:max_characters].rstrip() + "..."


def format_retrieved_context(
    retrieved_df: pd.DataFrame,
    max_chunks: int = 5,
    max_characters_per_chunk: int = 1200,
) -> str:
    """
    Format retrieved chunks into a structured context block for RAG prompting.
    """
    context_blocks = []

    for _, row in retrieved_df.head(max_chunks).iterrows():
        rank = row.get("rank", "")
        source_id = row.get("source_id", "")
        source_name = row.get("source_name", "")
        publisher = row.get("publisher", "")
        chunk_index = row.get("chunk_index", "")
        url = row.get("url", "")
        distance = row.get("distance", "")
        retrieved_text = truncate_text(
            row.get("retrieved_text", ""),
            max_characters=max_characters_per_chunk,
        )

        context_block = f"""
[Retrieved Chunk {rank}]
Source ID: {source_id}
Source Name: {source_name}
Publisher: {publisher}
Chunk Index: {chunk_index}
Retrieval Distance: {distance}
URL: {url}

Text:
{retrieved_text}
""".strip()

        context_blocks.append(context_block)

    return "\n\n" + ("-" * 80) + "\n\n".join(context_blocks)


print("Retrieved context formatting function is ready.")

Retrieved context formatting function is ready.


In [14]:
# Test retrieved context formatting

test_question = "What is the risk-based approach of the EU AI Act?"

test_retrieved_df = search_multisource_vector_store_source_aware(
    query=test_question,
    n_results=5,
    n_initial_results=20,
)

formatted_context = format_retrieved_context(
    retrieved_df=test_retrieved_df,
    max_chunks=4,
    max_characters_per_chunk=900,
)

print(f"Question: {test_question}")
print("=" * 100)
print(formatted_context)

Question: What is the risk-based approach of the EU AI Act?
[Retrieved Chunk 1]
Source ID: SRC-001
Source Name: AI Act
Publisher: European Commission
Chunk Index: 2
Retrieval Distance: 0.525324821472168
URL: https://digital-strategy.ec.europa.eu/en/policies/regulatory-framework-ai

Text:
The AI Act ensures that Europeans can trust what AI has to offer. While most AI systems pose limited to no risk and can contribute to solving many societal challenges, certain AI systems create risks that we must address to avoid undesirable outcomes. For example, it is often not possible to find out why an AI system has made a decision or prediction and taken a particular action. So, it may become difficult to assess whether someone has been unfairly disadvantaged, such as in a hiring decision or in an application for a public benefit scheme. Although existing legislation provides some protection, it is insufficient to address the specific challenges AI systems may bring. Risk-based Approach Complianc

## Responsible RAG Prompt

This section creates the prompt used by the final assistant.

The prompt instructs the assistant to:

- answer only from retrieved context
- avoid legal advice
- avoid unsupported claims
- cite retrieved source IDs and chunk indices
- clearly state when the available context is insufficient
- include a responsible-use note

This is an important guardrail for a Responsible AI / compliance-support RAG system.

In [11]:
def build_responsible_rag_prompt(
    question: str,
    formatted_context: str,
) -> str:
    """
    Build an LLM-ready RAG prompt with responsible-use guardrails.
    """
    prompt = f"""
You are a Responsible AI knowledge-support assistant.

You answer questions using only the retrieved public-source context provided below.

Important rules:
1. Do not provide legal advice.
2. Do not claim certainty beyond the retrieved context.
3. Do not use outside knowledge.
4. If the retrieved context is insufficient, say that the available sources do not support a confident answer.
5. Cite the source IDs and chunk indices used in the answer.
6. Mention that real compliance decisions should be reviewed by qualified legal, privacy, or compliance professionals.
7. Keep the answer clear, structured, and source-aware.

User question:
{question}

Retrieved context:
{formatted_context}

Answer format:

1. Short answer
Provide a concise answer grounded only in the retrieved context.

2. Source-grounded evidence
List the most relevant source IDs and chunk indices, with a short explanation of what each supports.

3. Limitations
State any limits of the retrieved context.

4. Responsible-use note
Clearly state that this is not legal advice.
""".strip()

    return prompt


print("Responsible RAG prompt builder is ready.")

Responsible RAG prompt builder is ready.


In [15]:
# Test RAG prompt construction

test_prompt = build_responsible_rag_prompt(
    question=test_question,
    formatted_context=formatted_context,
)

print(f"Prompt length in characters: {len(test_prompt)}")
print("=" * 100)
print(test_prompt[:5000])

Prompt length in characters: 5851
You are a Responsible AI knowledge-support assistant.

You answer questions using only the retrieved public-source context provided below.

Important rules:
1. Do not provide legal advice.
2. Do not claim certainty beyond the retrieved context.
3. Do not use outside knowledge.
4. If the retrieved context is insufficient, say that the available sources do not support a confident answer.
5. Cite the source IDs and chunk indices used in the answer.
6. Mention that real compliance decisions should be reviewed by qualified legal, privacy, or compliance professionals.
7. Keep the answer clear, structured, and source-aware.

User question:
What is the risk-based approach of the EU AI Act?

Retrieved context:
[Retrieved Chunk 1]
Source ID: SRC-001
Source Name: AI Act
Publisher: European Commission
Chunk Index: 2
Retrieval Distance: 0.525324821472168
URL: https://digital-strategy.ec.europa.eu/en/policies/regulatory-framework-ai

Text:
The AI Act ensures that Eu

In [13]:
def format_retrieved_context(
    retrieved_df: pd.DataFrame,
    max_chunks: int = 5,
    max_characters_per_chunk: int = 1200,
) -> str:
    """
    Format retrieved chunks into a structured context block for RAG prompting.
    """
    context_blocks = []

    for _, row in retrieved_df.head(max_chunks).iterrows():
        rank = row.get("rank", "")
        source_id = row.get("source_id", "")
        source_name = row.get("source_name", "")
        publisher = row.get("publisher", "")
        chunk_index = row.get("chunk_index", "")
        url = row.get("url", "")
        distance = row.get("distance", "")
        retrieved_text = truncate_text(
            row.get("retrieved_text", ""),
            max_characters=max_characters_per_chunk,
        )

        context_block = f"""
[Retrieved Chunk {rank}]
Source ID: {source_id}
Source Name: {source_name}
Publisher: {publisher}
Chunk Index: {chunk_index}
Retrieval Distance: {distance}
URL: {url}

Text:
{retrieved_text}
""".strip()

        context_blocks.append(context_block)

    separator = "\n\n" + ("-" * 80) + "\n\n"
    return separator.join(context_blocks)


print("Retrieved context formatting function updated.")

Retrieved context formatting function updated.


## LLM-Assisted Answer Generation

This section connects the retrieval and prompt-building layers to an LLM answer-generation workflow.

The notebook supports two modes:

1. **OpenAI generation mode**  
   If an `OPENAI_API_KEY` is available in the local `.env` file, the assistant can generate an LLM-assisted answer.

2. **Context-only fallback mode**  
   If no API key is available, the notebook still produces a structured source-grounded answer from the retrieved context.

This makes the project reproducible and safe for GitHub, because the notebook can run without exposing private API credentials.

In [23]:
from dotenv import load_dotenv

# Load local environment variables from .env if available
# Important: .env is ignored by .gitignore and should not be uploaded to GitHub.

env_path = PROJECT_ROOT / ".env"
load_dotenv(env_path)

openai_api_key_available = bool(os.getenv("OPENAI_API_KEY"))

# Model can be changed in .env later if needed:
# OPENAI_MODEL=gpt-5.5
llm_model_name = os.getenv("OPENAI_MODEL", "gpt-5.5")

print(f".env path: {env_path}")
print(f".env file exists: {env_path.exists()}")
print(f"OpenAI API key available: {openai_api_key_available}")
print(f"Configured LLM model: {llm_model_name}")

.env path: c:\Users\mdadg\Desktop\responsible-ai-rag-assistant\.env
.env file exists: True
OpenAI API key available: True
Configured LLM model: gpt-4.1


In [24]:
def estimate_retrieval_confidence(retrieved_df: pd.DataFrame) -> str:
    """
    Estimate retrieval confidence from top distance and source coverage.
    This is a lightweight project-level heuristic, not a formal metric.
    """
    if retrieved_df.empty:
        return "no_context"

    top_distance = float(retrieved_df["distance"].min())
    unique_sources = retrieved_df["source_id"].nunique()

    if top_distance <= 0.60 and unique_sources >= 1:
        return "moderate_to_good"

    if top_distance <= 0.80 and unique_sources >= 1:
        return "moderate"

    return "low"


def generate_context_only_answer(
    question: str,
    retrieved_df: pd.DataFrame,
    max_evidence_items: int = 4,
) -> str:
    """
    Generate a structured fallback answer without calling an external LLM.

    This answer is extractive/summarizing only.
    It does not use outside knowledge.
    """
    if retrieved_df.empty:
        return f"""
Question:
{question}

Context-grounded answer:
The available retrieved sources do not provide enough information to answer this question confidently.

Limitations:
No relevant context was retrieved from the vector store.

Responsible-use note:
This prototype does not provide legal advice. Real compliance decisions should be reviewed by qualified legal, privacy, or compliance professionals.
""".strip()

    confidence = estimate_retrieval_confidence(retrieved_df)

    evidence_lines = []

    for _, row in retrieved_df.head(max_evidence_items).iterrows():
        snippet = truncate_text(row["retrieved_text"], max_characters=350)

        evidence_lines.append(
            f"- Source {row['source_id']} — {row['source_name']}, "
            f"chunk {row['chunk_index']} "
            f"(distance: {row['distance']:.4f}): {snippet}"
        )

    source_reference_lines = []

    for _, row in retrieved_df.head(max_evidence_items).iterrows():
        source_reference_lines.append(
            f"- {row['source_id']} — {row['source_name']} "
            f"({row['publisher']}), chunk {row['chunk_index']}: {row['url']}"
        )

    answer = f"""
Question:
{question}

Context-grounded answer:
The retrieved context contains potentially relevant information for this question. The answer below should be treated as a source-grounded evidence summary, not as legal advice.

Retrieval confidence:
{confidence}

Key retrieved evidence:
{chr(10).join(evidence_lines)}

Source references:
{chr(10).join(source_reference_lines)}

Limitations:
This fallback answer is generated only from retrieved text snippets and does not perform full legal interpretation. The retrieved context may not cover all relevant legal provisions, exceptions, definitions, or implementation guidance.

Responsible-use note:
This prototype does not provide legal advice. Real compliance decisions should be reviewed by qualified legal, privacy, or compliance professionals.
""".strip()

    return answer


print("Context-only fallback answer function is ready.")

Context-only fallback answer function is ready.


In [25]:
def generate_openai_rag_answer(
    prompt: str,
    model: str = llm_model_name,
) -> str | None:
    """
    Generate an LLM-assisted RAG answer using the OpenAI Responses API.

    Returns None if:
    - no API key is available
    - the openai package is unavailable
    - the API call fails

    This allows the notebook to fall back safely to context-only generation.
    """
    if not os.getenv("OPENAI_API_KEY"):
        print("OpenAI API key not available. Using context-only fallback.")
        return None

    try:
        from openai import OpenAI

        client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

        response = client.responses.create(
            model=model,
            input=prompt,
        )

        return response.output_text

    except Exception as error:
        print("OpenAI generation failed. Using context-only fallback.")
        print(f"Error type: {type(error).__name__}")
        print(f"Error message: {error}")
        return None


print("OpenAI RAG generation function with fallback is ready.")

OpenAI RAG generation function with fallback is ready.


In [26]:
def answer_question_with_rag(
    question: str,
    n_results: int = 5,
    n_initial_results: int = 20,
    max_context_chunks: int = 5,
    max_characters_per_chunk: int = 1200,
    use_openai: bool = True,
) -> dict:
    """
    Full RAG pipeline:

    1. Infer preferred sources.
    2. Retrieve source-aware context.
    3. Format retrieved context.
    4. Build responsible RAG prompt.
    5. Generate answer with OpenAI if available.
    6. Fall back to context-only answer if OpenAI is unavailable.
    """
    retrieved_df = search_multisource_vector_store_source_aware(
        query=question,
        n_results=n_results,
        n_initial_results=n_initial_results,
    )

    formatted_context = format_retrieved_context(
        retrieved_df=retrieved_df,
        max_chunks=max_context_chunks,
        max_characters_per_chunk=max_characters_per_chunk,
    )

    rag_prompt = build_responsible_rag_prompt(
        question=question,
        formatted_context=formatted_context,
    )

    llm_answer = None

    if use_openai:
        llm_answer = generate_openai_rag_answer(
            prompt=rag_prompt,
            model=llm_model_name,
        )

    if llm_answer:
        answer_method = "openai_responses_api"
        final_answer = llm_answer
    else:
        answer_method = "context_only_fallback"
        final_answer = generate_context_only_answer(
            question=question,
            retrieved_df=retrieved_df,
        )

    result = {
        "question": question,
        "preferred_sources": infer_preferred_sources(question),
        "answer_method": answer_method,
        "retrieved_chunks": len(retrieved_df),
        "retrieved_sources": retrieved_df["source_id"].unique().tolist(),
        "top_distance": float(retrieved_df["distance"].min()) if not retrieved_df.empty else None,
        "formatted_context": formatted_context,
        "rag_prompt": rag_prompt,
        "answer": final_answer,
        "retrieved_df": retrieved_df,
    }

    return result


print("Unified RAG assistant function is ready.")

Unified RAG assistant function is ready.


In [27]:
# Test the full RAG assistant pipeline

final_test_question = "What is the risk-based approach of the EU AI Act?"

rag_result = answer_question_with_rag(
    question=final_test_question,
    n_results=5,
    n_initial_results=20,
    max_context_chunks=5,
    max_characters_per_chunk=1000,
    use_openai=True,
)

print(f"Question: {rag_result['question']}")
print(f"Preferred sources: {rag_result['preferred_sources']}")
print(f"Answer method: {rag_result['answer_method']}")
print(f"Retrieved chunks: {rag_result['retrieved_chunks']}")
print(f"Retrieved sources: {rag_result['retrieved_sources']}")
print(f"Top distance: {rag_result['top_distance']}")
print("=" * 100)
print(rag_result["answer"])

print("=" * 100)
display(
    rag_result["retrieved_df"][
        [
            "rank",
            "distance",
            "source_id",
            "source_name",
            "publisher",
            "chunk_index",
            "word_count",
            "preferred_source",
        ]
    ]
)

OpenAI generation failed. Using context-only fallback.
Error type: RateLimitError
Error message: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}
Question: What is the risk-based approach of the EU AI Act?
Preferred sources: ['SRC-001', 'SRC-002']
Answer method: context_only_fallback
Retrieved chunks: 5
Retrieved sources: ['SRC-001']
Top distance: 0.525324821472168
Question:
What is the risk-based approach of the EU AI Act?

Context-grounded answer:
The retrieved context contains potentially relevant information for this question. The answer below should be treated as a source-grounded evidence summary, not as legal advice.

Retrieval confidence:
moderate_to_good

Key retrieved evidence:
- Source SRC-001 — AI Act, chunk 2 (dista

,rank,distance,source_id,source_name,publisher,chunk_index,word_count,preferred_source
0,1,0.525325,SRC-001,AI Act,European Commission,2,300,True
1,2,0.537072,SRC-001,AI Act,European Commission,1,300,True
2,3,0.735104,SRC-001,AI Act,European Commission,8,300,True
3,4,0.800995,SRC-001,AI Act,European Commission,7,300,True
4,5,0.810517,SRC-001,AI Act,European Commission,3,300,True


## API Quota Handling and Fallback Mode

The OpenAI API key was loaded successfully from the local `.env` file, but the API request returned a quota/billing error during testing.

This is handled safely by the project design.

If OpenAI generation is unavailable because of missing credentials, quota limits, billing limits, network problems, or API errors, the assistant automatically switches to a context-only fallback answer.

This fallback mode:

- still uses the multi-source Chroma vector store
- still uses source-aware retrieval
- still cites retrieved sources and chunk indices
- avoids unsupported claims
- clearly states that it does not provide legal advice
- keeps the notebook reproducible without paid API access

For the portfolio version, OpenAI generation is treated as an optional enhancement, while the fallback mode remains the reliable default.

In [28]:
# Project-level generation setting
# Keep this False for the reproducible portfolio version without paid API usage.
# If API billing/quota is enabled later, this can be changed to True.

ENABLE_OPENAI_GENERATION = False

print(f"OpenAI API key available: {openai_api_key_available}")
print(f"OpenAI generation enabled for this run: {ENABLE_OPENAI_GENERATION}")

if openai_api_key_available and not ENABLE_OPENAI_GENERATION:
    print("API key is detected, but OpenAI generation is intentionally disabled for reproducible fallback-mode testing.")

OpenAI API key available: True
OpenAI generation enabled for this run: False
API key is detected, but OpenAI generation is intentionally disabled for reproducible fallback-mode testing.


In [29]:
# Test final RAG pipeline in fallback mode without calling OpenAI

fallback_test_question = "What is the risk-based approach of the EU AI Act?"

fallback_rag_result = answer_question_with_rag(
    question=fallback_test_question,
    n_results=5,
    n_initial_results=20,
    max_context_chunks=5,
    max_characters_per_chunk=1000,
    use_openai=ENABLE_OPENAI_GENERATION,
)

print(f"Question: {fallback_rag_result['question']}")
print(f"Preferred sources: {fallback_rag_result['preferred_sources']}")
print(f"Answer method: {fallback_rag_result['answer_method']}")
print(f"Retrieved chunks: {fallback_rag_result['retrieved_chunks']}")
print(f"Retrieved sources: {fallback_rag_result['retrieved_sources']}")
print(f"Top distance: {fallback_rag_result['top_distance']}")
print("=" * 100)
print(fallback_rag_result["answer"])

print("=" * 100)
display(
    fallback_rag_result["retrieved_df"][
        [
            "rank",
            "distance",
            "source_id",
            "source_name",
            "publisher",
            "chunk_index",
            "word_count",
            "preferred_source",
        ]
    ]
)

Question: What is the risk-based approach of the EU AI Act?
Preferred sources: ['SRC-001', 'SRC-002']
Answer method: context_only_fallback
Retrieved chunks: 5
Retrieved sources: ['SRC-001']
Top distance: 0.525324821472168
Question:
What is the risk-based approach of the EU AI Act?

Context-grounded answer:
The retrieved context contains potentially relevant information for this question. The answer below should be treated as a source-grounded evidence summary, not as legal advice.

Retrieval confidence:
moderate_to_good

Key retrieved evidence:
- Source SRC-001 — AI Act, chunk 2 (distance: 0.5253): The AI Act ensures that Europeans can trust what AI has to offer. While most AI systems pose limited to no risk and can contribute to solving many societal challenges, certain AI systems create risks that we must address to avoid undesirable outcomes. For example, it is often not possible to find out why an AI system has made a decision or predicti...
- Source SRC-001 — AI Act, chunk 1 (dist

,rank,distance,source_id,source_name,publisher,chunk_index,word_count,preferred_source
0,1,0.525325,SRC-001,AI Act,European Commission,2,300,True
1,2,0.537072,SRC-001,AI Act,European Commission,1,300,True
2,3,0.735104,SRC-001,AI Act,European Commission,8,300,True
3,4,0.800995,SRC-001,AI Act,European Commission,7,300,True
4,5,0.810517,SRC-001,AI Act,European Commission,3,300,True


In [30]:
# Batch-test the final RAG assistant in reproducible fallback mode

final_rag_test_questions = [
    {
        "test_id": "FINAL-RAG-001",
        "question": "What is the risk-based approach of the EU AI Act?",
        "expected_source_family": "EU AI Act",
    },
    {
        "test_id": "FINAL-RAG-002",
        "question": "What obligations apply to high-risk AI systems?",
        "expected_source_family": "EU AI Act",
    },
    {
        "test_id": "FINAL-RAG-003",
        "question": "What is personal data under the GDPR?",
        "expected_source_family": "GDPR",
    },
    {
        "test_id": "FINAL-RAG-004",
        "question": "What does NIST say about managing AI risks?",
        "expected_source_family": "NIST AI RMF",
    },
]

final_rag_test_records = []

for item in final_rag_test_questions:
    result = answer_question_with_rag(
        question=item["question"],
        n_results=5,
        n_initial_results=20,
        max_context_chunks=5,
        max_characters_per_chunk=1000,
        use_openai=ENABLE_OPENAI_GENERATION,
    )

    answer_preview = truncate_text(result["answer"], max_characters=500)

    final_rag_test_records.append(
        {
            "test_id": item["test_id"],
            "question": item["question"],
            "expected_source_family": item["expected_source_family"],
            "preferred_sources": ", ".join(result["preferred_sources"]),
            "answer_method": result["answer_method"],
            "retrieved_chunks": result["retrieved_chunks"],
            "retrieved_sources": ", ".join(result["retrieved_sources"]),
            "top_distance": result["top_distance"],
            "answer_preview": answer_preview,
        }
    )

final_rag_test_results_df = pd.DataFrame(final_rag_test_records)

display(final_rag_test_results_df)

,test_id,question,expected_source_family,preferred_sources,answer_method,retrieved_chunks,retrieved_sources,top_distance,answer_preview
0,FINAL-RAG-001,What is the risk-based approach of the EU AI Act?,EU AI Act,"SRC-001, SRC-002",context_only_fallback,5,SRC-001,0.525325,Question: What is the risk-based approach of t...
1,FINAL-RAG-002,What obligations apply to high-risk AI systems?,EU AI Act,"SRC-001, SRC-002",context_only_fallback,5,"SRC-001, SRC-004",0.577118,Question: What obligations apply to high-risk ...
2,FINAL-RAG-003,What is personal data under the GDPR?,GDPR,SRC-003,context_only_fallback,5,SRC-003,0.807489,Question: What is personal data under the GDPR...
3,FINAL-RAG-004,What does NIST say about managing AI risks?,NIST AI RMF,"SRC-004, SRC-005",context_only_fallback,5,"SRC-005, SRC-004",0.403003,Question: What does NIST say about managing AI...


In [31]:
# Save final fallback-mode RAG test results

final_rag_test_results_path = PROCESSED_DATA_DIR / "final_rag_fallback_test_results.csv"

final_rag_test_results_df.to_csv(
    final_rag_test_results_path,
    index=False,
    encoding="utf-8",
)

print(f"Saved final fallback RAG test results to: {final_rag_test_results_path}")
print(f"File exists: {final_rag_test_results_path.exists()}")
print(f"Rows saved: {len(final_rag_test_results_df)}")

Saved final fallback RAG test results to: c:\Users\mdadg\Desktop\responsible-ai-rag-assistant\data\processed\final_rag_fallback_test_results.csv
File exists: True
Rows saved: 4


## Notebook 06 Final Checkpoint

This checkpoint confirms that the final RAG assistant layer works in reproducible fallback mode.

The notebook has now:

- loaded the multi-source Chroma vector store
- loaded the source-aware retrieval logic
- formatted retrieved context for RAG prompting
- built a responsible RAG prompt template
- tested optional OpenAI generation handling
- validated fallback-mode answer generation
- saved final fallback RAG test results

For the portfolio version, fallback mode is kept as the reliable default. OpenAI generation can be enabled later if API billing/quota is available.

In [32]:
# Final Notebook 06 checkpoint

notebook_06_checkpoint = {
    "openai_api_key_available": openai_api_key_available,
    "openai_generation_enabled": ENABLE_OPENAI_GENERATION,
    "configured_llm_model": llm_model_name,
    "default_answer_method": final_rag_test_results_df["answer_method"].unique().tolist(),
    "final_rag_test_rows": len(final_rag_test_results_df),
    "final_rag_test_file_exists": final_rag_test_results_path.exists(),
    "unique_expected_source_families": final_rag_test_results_df["expected_source_family"].nunique(),
    "unique_retrieved_source_sets": final_rag_test_results_df["retrieved_sources"].nunique(),
}

print("Notebook 06 final checkpoint completed.")
print("-" * 80)

for key, value in notebook_06_checkpoint.items():
    print(f"{key}: {value}")

Notebook 06 final checkpoint completed.
--------------------------------------------------------------------------------
openai_api_key_available: True
openai_generation_enabled: False
configured_llm_model: gpt-4.1
default_answer_method: ['context_only_fallback']
final_rag_test_rows: 4
final_rag_test_file_exists: True
unique_expected_source_families: 3
unique_retrieved_source_sets: 4


## Next Project Step

The next step is to turn the notebook-based RAG pipeline into a reusable project component.

The next notebook or script will focus on:

1. moving the final RAG assistant functions into reusable Python modules
2. creating a simple command-line or Streamlit interface
3. allowing users to ask questions interactively
4. displaying retrieved sources and answer confidence
5. keeping responsible-use guardrails visible
6. preparing the project for GitHub presentation

The core RAG system is now functional in reproducible fallback mode, with optional OpenAI generation support available when API quota is enabled.